# fnname: LoRA fine-tune on a free Kaggle GPU

**Before running:**
1. Upload `data/train.jsonl`, `data/val.jsonl` and `data/test.jsonl` as a Kaggle Dataset named **`fnname-data`** (Datasets → New Dataset). It's free and about 210 MB.
2. In this notebook: **Add Input** → your `fnname-data` dataset.
3. **Settings → Accelerator → GPU T4 x2** (free, ~30 h/week). **Settings → Internet → On.**
4. **Run All.** Training takes about 2–3 hours, and prediction about 20 minutes.

If the session dies, run all cells again. Training resumes from the last checkpoint saved in `/kaggle/working`.

In [ ]:
%cd /kaggle/working
# Needs Settings -> Internet -> On (otherwise this fails and nothing below can import fnname).
!test -d repo && git -C repo pull -q || git clone -q https://github.com/keshav0045/stripped-binary.git repo
%cd /kaggle/working/repo
!pip install -q peft datasets accelerate
# Kaggle preinstalls an old torchao that current peft refuses to load; LoRA doesn't need it.
!pip uninstall -q -y torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, shutil
os.makedirs("data", exist_ok=True)
# Kaggle mounts datasets at varying depths (/kaggle/input/<name>/ or deeper), so search recursively.
src = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
if not src:
    !find /kaggle/input -maxdepth 5 | head -30
assert src, "train.jsonl not found under /kaggle/input: attach fnname-data (Add Input); listing above"
for name in ("train.jsonl", "val.jsonl", "test.jsonl"):
    shutil.copy(os.path.join(os.path.dirname(src[0]), name), "data/")
!wc -l data/*.jsonl

In [ ]:
# Resuming a stopped run: Add Input -> Notebook Output -> pick the earlier version of this notebook.
# Its newest checkpoint is copied here and --resume continues from it. Fresh run: nothing to copy.
import glob, os, shutil
ckpts = sorted(glob.glob("/kaggle/input/**/lora/checkpoint-*", recursive=True), key=lambda p: int(p.rsplit("-", 1)[1]))
if ckpts and not glob.glob("/kaggle/working/lora/checkpoint-*"):
    shutil.copytree(ckpts[-1], os.path.join("/kaggle/working/lora", os.path.basename(ckpts[-1])))
print("resuming from", ckpts[-1] if ckpts else "scratch")


In [ ]:
# 30K random training functions, 1 epoch, 1K-token inputs: fits one free session.
# Checkpoints go to /kaggle/working so they survive a restart.
# batch 1 per GPU: the logits over Qwen's 152K-token vocab are ~1.2 GB per 1K-token example,
# and with T4 x2 they are gathered on GPU 0. 1 x 16 accumulation x 2 GPUs = 32 per step.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m fnname.llm.train_lora --base Qwen/Qwen2.5-Coder-1.5B --limit 30000 --max-len 1024 \
    --batch 1 --grad-accum 16 --out /kaggle/working/lora --resume

In [ ]:
# Same 1,000-function test sample (seed 0) as the local Ollama baseline, so results compare directly.
!python -m fnname.llm.predict_lora --adapter /kaggle/working/lora --test data/test.jsonl \
    --sample 1000 --batch 16 --out /kaggle/working/lora.jsonl

In [ ]:
# Download these two from the Output panel (right side) when it finishes:
#   lora.jsonl       -> put in preds/ on your Mac
#   lora_adapter.zip -> unzip into models/lora on your Mac (for serving the model)
!cd /kaggle/working && rm -rf lora/checkpoint-* && zip -qr lora_adapter.zip lora && ls -lh lora.jsonl lora_adapter.zip